#  Question 7

Extend the gold layer with a second aggregation for a different stakeholder (e.g., the inventory team) and justify why it belongs in gold rather than being computed ad hoc by that team.

In [0]:
-- Product Sales & Quantity Summary

CREATE OR REPLACE VIEW cyntexa_dev.medallion_day_6.product_quantity_summary AS
SELECT
    product_id,
    SUM(quantity) AS total_units_sold,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(total_amount) AS total_sales,
    AVG(quantity) AS avg_units_per_order
FROM cyntexa_dev.medallion_day_6.silver_sales
GROUP BY product_id;

### Justification
The inventory_summary aggregation is created in the Gold layer for the Inventory Team. Although the team could access the Silver table and calculate these metrics themselves, the requirement for product-level sales and quantity analysis is already known. Therefore, providing a Gold-level aggregation is more appropriate because the business logic is calculated centrally and can be reused by inventory dashboards and reports. It also ensures that all consumers use the same definitions for metrics such as total units sold and total orders, rather than implementing their own ad-hoc calculations on the Silver data.

The Silver layer remains available for detailed and flexible analysis when required, while the Gold layer provides optimized, business-ready metrics for the known Inventory Team use case.

# Question 8


The pipeline should perform data processing and transformations in the customer's Databricks data plane. This includes Bronze ingestion, Silver cleaning and transformation, and Gold aggregations and KPI calculations. Keeping these workloads in the data plane ensures that customer data remains within the customer's controlled environment while the Spark compute performs the actual processing.

The Databricks control plane should primarily be used for workspace management, job orchestration, scheduling, configuration, and other control functions. Customer data should not be unnecessarily processed or stored in the control plane.

For network and security, we need to make sure that only required connections are allowed between the customer's data plane and Databricks services.

We should also check:

* Who can access the data.
* Whether data is encrypted.
* Whether secrets are stored securely.
* Whether user and job activities are logged.
* Whether the customer's data stays isolated.

This keeps the actual customer data processing in the customer's environment while Databricks handles the control and management tasks.



# Question 9

In [0]:
CREATE OR REPLACE TABLE cyntexa_dev.medallion_day_6.gold_sales_total_revenue_per_customers_dq
AS
SELECT *
FROM cyntexa_dev.medallion_day_6.gold_sales_total_revenue_per_customers;

In [0]:
INSERT INTO cyntexa_dev.medallion_day_6.gold_sales_total_revenue_per_customers_dq
(customer_id, total_revenue)
VALUES
(NULL, 15000.00);

In [0]:
SELECT *
FROM cyntexa_dev.medallion_day_6.gold_sales_total_revenue_per_customers_dq
ORDER BY customer_id;

![](/Volumes/cyntexa_dev/medallion_day_6/raw2/bad_chart.png)

In [0]:
delete from cyntexa_dev.medallion_day_6.gold_sales_total_revenue_per_customers_dq where customer_id is null

![](/Volumes/cyntexa_dev/medallion_day_6/raw2/corrected_record.png)

## Steps

1. **Created the Gold KPI**

   * Created the **Customer-wise Total Revenue** KPI.
   * Grouped the data by `customer_id` and calculated `total_revenue`.

2. **Performed Data Quality Check**

   * Checked the KPI table to verify that `customer_id` was not NULL.
   * Found one record where `customer_id = NULL`, while revenue was still being calculated for that record.

3. **Created the Bad Visual**

   * Created a bar chart showing **customer-wise total revenue**.
   * The visual displayed the NULL customer as well, confirming the data quality issue. Ref- cell - 10

4. **Corrected the Gold KPI**

   * Corrected the invalid record with the NULL `customer_id` in the Gold KPI.
   * Kept all existing valid customer records unchanged.

5. **Refreshed the KPI and Visual**

   * Refreshed the KPI and the associated visual after applying the correction.
   * The visual was automatically updated with the corrected data.
   Ref. cell - 12

6. **Validated the Result**

   * Verified that the NULL customer ID no longer appears in the updated visual.
   * Customer-wise revenue is now displayed only against valid customer IDs.
